Ce notebook n'est pas fonctionnel. \
C'est un notebook qui a pour vocation d'expliquer comment le modèle fonctionne et est entraîné, il est possible d'essayer de l'entraîner à l'aide du CLI en téléchargeant un dataset dont les liens sont sur le Readme. Cependant l'entraînement peut s'avérer coûteux.

Notre modèle se base originellement sur l'architecture GeoCLIP dont le papier se trouve : [ici](https://arxiv.org/abs/2309.16020), mais nous différons sur beaucoup de points. Notamment sur l'utilisation d'images satellites, l'utilisation de DinoV2, notre loss et des adapteurs. \
Nous avons 2 versions du modèle, une qui utilise des images satellites, une autre sans. Nous allons parler de la version avec, la version sans étant analogue. \
Nous avons donc 3 encodeurs :
- Un encodeur de positions qui est une fonction $f_{loc} : pos \rightarrow Embed$
- Un encodeur d'images qui est une fonction $f_{img} : img \rightarrow Embed$
- Un encodeur d'images satellites qui est une fonction $f_{sat} : sat \rightarrow Embed$

Les espaces latents sont les mêmes et l'objectif est de faire coincider les encodeurs sur un même espace latent. Pour ensuite avoir un encodeur d'image correspondant à celui de positions, l'encodeur dédié aux images satellites étant surtout présent en tant que "distillateur" de connaissances, pour faire correspondre les images et donner implicitement plus de contexte, sans pour autant en nécessiter lors de l'inférence.\
Pour cela on utilise une méthode d'apprentissage par contraste, on va faire prédire un batch, puis mettre en relief, on va calculer une matrice de similarité et récompenser lorsque que les éléments à prédire sont ceux ayant le meilleur score. Ici l'on fait ça pour : img <-> sat, sat <-> loc, loc <-> img. \
\
La tâche de la géolocalisation étant remarquablement difficile, surtout dans le cadre "naturel" dans lequel nous nous plaçons (c'est à dire avec des données uniformes), nous avons fait le choix d'utiliser un modèle de fondation, c'est à dire un modèle pré-entraîné que nous pouvons ensuite utiliser pour nos tâches. Le modèle de fondation ici est DinoV2 en son format "Small", que nous allons geler pour ne pas toucher à ses poids.

### Partie 0 : Le dataset

Tous nos datasets sont issus de Google Streeview (scrappé par nos soins à l'aide de [streetlevel](https://github.com/sk-zk/streetlevel)).

Nous avons 3 datasets :
- Un dataset en France centré sur 50 villes, il contient 70K images pour 35K positions.
- Un dataset en France, cette fois-ci principalement uniforme (avec un certain focus sur les villes, mais à moindre échelle), il contient 300K images pour 150K positions.
- Un dataset sur Paris, de nouveau uniforme mais dans une zone plus restreinte (il ne va pas beaucoup plus loin que La Défense et Cachan). Il a également en complément, pour chaque image, son image satellite associée.


## Partie 1 : le modèle

### L'encodeur d'images

L'encodeur d'images contient deux parties :
- Le backbone (avec deux adapteurs)
  - L'on utilise le modèle de fondation [DinoV2](https://github.com/facebookresearch/dinov2) en sa version "small", il est gelé et on ne l'utilise que pour de l'inférence
  - et deux têtes d'adapteurs, une pour chaque type de feature que l'on récupère de DinoV2 (cls_token qui va "résumer" l'image et les patch_tokens qui sont les features pour chaque patch de l'image)
    - L'idée des adapteurs vient d'[ici](https://arxiv.org/pdf/1902.00751/1000) mais aussi de la méthode LoRA, nous ne pouvions appliquer aucune des deux dans nos limitations donc nous en avons mis un en sortie. 
  - On applique également une couche de [GeM Pooling](https://amaarora.github.io/posts/2020-08-30-gempool.html) qui est un équilibre entre le max pooling et l'avg pooling.
- On utilise une tête de projection qui n'est qu'un MLP à 2 couches cachées.


#### Cet encodeur est utilisé pour les images StreetView ET pour les images satellites !

Le code des adapteurs :
```python
class FeatureAdapter(nn.Module):
    def __init__(self, in_dim=384, bottleneck=96):
        super().__init__()
        self.down = nn.Linear(in_dim, bottleneck)
        self.act = nn.GELU()
        self.up = nn.Linear(bottleneck, in_dim)
        self.scale = nn.Parameter(torch.ones(1) * 0.1)
    
    def forward(self, x):
        out = self.down(x)
        out = self.act(out)
        out = self.up(out)
        return x + self.scale * adapted
```

Le code de la projection : 
```python
self.proj = nn.Sequential(
    nn.Linear(768, 2048),
    nn.LayerNorm(2048),
    nn.GELU(),
    #On a très peu de données, donc on utilise du dropout pour éviter l'overfitting instant
    nn.Dropout(0.3),
    nn.Linear(2048, 1024),
    nn.LayerNorm(1024),
    nn.GELU(),
    nn.Dropout(0.3),
    nn.Linear(1024, 512)
)
```

La forward se passe alors comme :
```python
#On applique dino sur notre image, on récupère les features
with torch.no_grad():
    output = self.backbone.forward_features(x)

#On récupère les deux types de features
patch_tokens = self.patch_adapter(output["x_norm_patchtokens"])
cls_token = self.cls_adapter(output["x_norm_clstoken"])

#On applique le GeM pooling
pooled_patches = self.pool(patch_tokens)

#L'on combine pour pouvoir projeter !
combined = torch.cat([pooled_patches, cls_token], dim=1)

embeddings = self.proj(combined)
return F.normalize(embeddings, p=2, dim=1)
```

### L'encodeur de positions
L'encodeur de positions est plus simple mais a plus de paramètres, il est très inspiré de l'architecture GeoCLIP, qui est lui même une architecture classique :
- On projette les coordonnées, ici l'on normalise en fonction du rectangle dans lesquels les données ont été prises, donc la France ou Paris
- On va ensuite encoder les positions à l'aide [Random Fourier Features](https://github.com/jmclong/random-fourier-features-pytorch), ce qui permet au modèle d'apprendre sur des fonctions continues (et à plusieurs échelles !) et ne pas avoir à subir de fortes variations (en basse dimension). C'est un choix naturel surtout que nous essayons d'approximer une fonction continue, il existe aussi d'autres alternatives plus modernes comme [SIREN](https://arxiv.org/pdf/2006.09661) ou il semblerait même que les deux se [mélangent](https://xeonqq.github.io/machine%20learning/fourier-feature-siren/). Cependant nous n'avons pas réussi à faire converger un modèle utilisant SIREN.
- Puis pour chaque échelle il y a un block résiduel constitué d'un MLP
- On projette finalement vers une dimension de 512 en une couche pleinement connectée

Le code des blocs résiduels est :
```python
class ResBlock(nn.Module):
    def __init__(self, hidden_dim, encoded_size=256, dropout=0.1):
        super().__init__()
        self.rff_proj = nn.Linear(encoded_size * 2, hidden_dim)
        self.net = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim) 
        )
    def forward(self, current_state, rff_feat):
        freq_emb = self.rff_proj(rff_feat)
        x = current_state + freq_emb
        out = self.net(x)
        return x + out
```

Le location encoder est alors :
```python
self.sigmas = [2.0, 16.0, 64.0, 256.0]

encoded_size = 256
hidden_dim = 1024 

self.rff_layers = nn.ModuleList([
    rff.layers.GaussianEncoding(sigma=s, input_size=2, encoded_size=encoded_size)
    for s in self.sigmas
])

self.blocks = nn.ModuleList([
    ResBlock(hidden_dim, encoded_size, dropout=0.1)
    for _ in self.sigmas
])

self.start_token = nn.Parameter(torch.randn(1, hidden_dim))
self.final_proj = nn.Linear(hidden_dim, 512)
```

### L'encodeur global
Ce n'est alors qu'un encapsuleur des deux encodeurs précédents avec un paramètre de scale utilisé dans la loss, permettant d'aider l'apprentissage.
```python
class MixedEncoder(nn.Module):
    def __init__(self):
        super(MixedEncoder, self).__init__()
        self.image_encoder = ImageEncoder()
        self.location_encoder = LocationEncoder()
        self.logit_scale = nn.Parameter(torch.ones([]) * 2.6592)

    def forward(self, img, loc):
        img_embed = self.image_encoder(img)
        loc_embed = self.location_encoder(loc)
        return img_embed, loc_embed, self.logit_scale.exp()
```

## Partie 2 : l'apprentissage

Notre boucle d'apprentissage est assez simple, nous avons des triplets image/position/sat et nous utilisons une méthode d'apprentissage contrastif (Nous avons également essayé d'utiliser une triplet margin loss sur une architecture où nous donnions uniquement les images sans les positions, sans grand succès).

### Les données lors de l'apprentissage

Nous devons alors retourner des couples img/position, nous utilisons des augmentations sur les images afin de s'assurer que le modèle ne se focus pas sur des détails tels que la saison et nous ajoutons également un (léger) bruit aux coordonnées lors de l'entraînement. Dans l'espoir d'obtenir une meilleur robustesse.

### La loss

Soit $M_{i,j}$ la matrice des dupliqués (à epsilon près), une case est à $1$ si $i = j$ ou si la position $i$ est à distance epsilon de la position $j$. \
Plus formellement cela s'écrit : 
$
M_{i,j} = 
    \begin{cases} 
        1 & \text{si } i = j \\
        0 & \text{si } i \neq j \text{ et } \|\text{pos}_i - \text{pos}_j\| < \epsilon \\
        1 & \text{sinon}
    \end{cases} $ \
On définit alors la [Loss InfoNCE](https://lilianweng.github.io/posts/2021-05-31-contrastive/#infonce) masquée (c'est à dire prenant compte des doublons) entre deux types d'entrées (img, sat ou loc) comme : $\mathcal{L}_{\text{NCE}}(\mathbf{X}, \mathbf{Y})$ \
On peut alors définir la loss symétrique : $\mathcal{L}_{\text{sym}}(\mathbf{X}, \mathbf{Y}) = \frac{1}{2} \left[ \mathcal{L}_{\text{NCE}}(\mathbf{X}, \mathbf{Y}) + \mathcal{L}_{\text{NCE}}(\mathbf{Y}, \mathbf{X}) \right]$ \
Et l'on a alors une loss symétrique pour chaque paire de types d'entrées, on a lors : 
    $\mathcal{L}_{\text{total}} = \lambda_1 \mathcal{L}_{\text{sym}}(\mathbf{v}^{\text{img}}, \mathbf{v}^{\text{loc}}) 
    + \lambda_2 \mathcal{L}_{\text{sym}}(\mathbf{v}^{\text{img}}, \mathbf{v}^{\text{sat}}) 
    + \lambda_3 \mathcal{L}_{\text{sym}}(\mathbf{v}^{\text{sat}}, \mathbf{v}^{\text{loc}})$ \
Où les $\alpha_i$ sont les différents coefficients (ici généralement (0.4,0.4,0.2))

L'intuition derrière cette loss est qu'elle va récompenser les bons guess X <-> Y mais punir les mauvais. Cette loss est très dépendante de la taille des batch, certains modèles CLIP vont jusqu'à des batch de taille 32000. \
\
Mais en raison de contraintes matérielles notre méthode est :
- Nous utilisons des plus petits batch (32 généralement)
- Nous accumulons les résultats des petits batchs **sans faire de loss ou de backward** tant que l'on a moins de 1024 éléments
- Nous calculons la loss sur les 1024 accumulés et nous faisons enfin une mise à jour du réseau

Ce qui nous permet de profiter de la loss constrastive à plus petits coût, au prix d'une actualisation tous les 32 batches.
\
Il existe aussi d'autres méthodes (des heuristiques) pour gérer le besoin de gros batches de la loss contrastive, mais comme nous avons un modèle simple et que DinoV2 est gelé, nous pouvons nous permettre de ne pas utiliser d'heuristique.

## Partie 3 : Le retrieval
Une fois le modèle entraîné, il faut créer une base de données pour ensuite pouvoir faire du retrieval. \
Il existe plusieurs méthodes, utiliser une grille (que l'on encode à l'aide de $f_pos$), faire un sampling du dataset. \
Nous utilisons un sampling du dataset ici. Et nous appliquons un algorithme de [k-Nearest Neighbors](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm). Nous utilisions une structure de KMeans hiérarchique, mais étant une structure probabiliste et étant donné que nous avons assez peu de données, nous l'avons supprimée.